# Topology Parallel Benchmark

This notebook demonstrates the current maintained CLI commands on lightweight benchmark cases.

Relevant docs:
- [docs/problems/Topology.md](../../docs/problems/Topology.md)
- [docs/results/Topology.md](../../docs/results/Topology.md)


In [1]:
from pathlib import Path
import json
import subprocess

REPO_ROOT = Path.cwd()
ARTIFACT_ROOT = Path('artifacts/raw_results/notebook_runs')
(REPO_ROOT / ARTIFACT_ROOT).mkdir(parents=True, exist_ok=True)
PYTHON = './.venv/bin/python'

def dump_json(path):
    with open(REPO_ROOT / path, encoding='utf-8') as handle:
        return json.load(handle)


In [2]:
from pprint import pprint

out_dir = ARTIFACT_ROOT / 'topology_parallel_benchmark'
(REPO_ROOT / out_dir).mkdir(parents=True, exist_ok=True)

serial_json = out_dir / 'serial_reference.json'
serial_cmd = [
    PYTHON,
    'src/problems/topology/jax/solve_topopt_jax.py',
    '--nx', '64', '--ny', '32',
    '--fixed_pad_cells', '4', '--load_pad_cells', '4',
    '--outer_maxit', '12', '--mechanics_maxit', '40', '--design_maxit', '40',
    '--quiet', '--json_out', str(serial_json),
]
print('$', ' '.join(serial_cmd))
subprocess.run(serial_cmd, cwd=REPO_ROOT, check=True)

parallel_json = out_dir / 'parallel_reference.json'
parallel_cmd = [
    'mpiexec', '-n', '2', PYTHON,
    'src/problems/topology/jax/solve_topopt_parallel.py',
    '--nx', '64', '--ny', '32',
    '--fixed_pad_cells', '4', '--load_pad_cells', '4',
    '--outer_maxit', '12', '--design_maxit', '8',
    '--quiet', '--json_out', str(parallel_json),
]
print('$', ' '.join(parallel_cmd))
subprocess.run(parallel_cmd, cwd=REPO_ROOT, check=True)

serial_payload = dump_json(serial_json)
parallel_payload = dump_json(parallel_json)
summary = {
    'serial_result': serial_payload.get('result'),
    'serial_compliance': serial_payload.get('final_compliance'),
    'parallel_result': parallel_payload.get('result'),
    'parallel_compliance': parallel_payload.get('final_compliance'),
    'parallel_outer_iterations': parallel_payload.get('outer_iterations_completed'),
}
pprint(summary)


$ ./.venv/bin/python src/problems/topology/jax/solve_topopt_jax.py --nx 64 --ny 32 --fixed_pad_cells 4 --load_pad_cells 4 --outer_maxit 12 --mechanics_maxit 40 --design_maxit 40 --quiet --json_out artifacts/raw_results/notebook_runs/topology_parallel_benchmark/serial_reference.json


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{
  "solver": "pure_jax_staggered_topopt",
  "backend": "serial",
  "result": "max_outer_iterations",
  "time": 1.672980458009988,
  "setup_time": 0.8532006710302085,
  "nprocs": 1,
  "mesh": {
    "nx": 64,
    "ny": 32,
    "nodes": 2145,
    "elements": 4096,
    "displacement_free_dofs": 4224,
    "design_free_dofs": 1905
  },
  "parameters": {
    "length": 2.0,
    "height": 1.0,
    "traction": 1.0,
    "load_fraction": 0.2,
    "fixed_pad_cells": 4,
    "load_pad_cells": 4,
    "volume_fraction_target": 0.4,
    "theta_min": 0.001,
    "solid_latent": 10.0,
    "young": 1.0,
    "poisson": 0.3,
    "alpha_reg": 0.005,
    "ell_pf": 0.08,
    "mu_move": 0.01,
    "lambda_init": 0.0,
    "beta_lambda": 12.0,
    "volume_penalty": 10.0,
    "p_start": 1.0,
    "p_max": 4.0,
    "p_increment": 0.5,
    "continuation_interval": 20,
    "outer_maxit": 12,
    "outer_tol": 0.02,
    "volume_tol": 0.001
  },
  "solver_options": {
    "mechanics_solver_type": "petsc_gamg",
    "mechanic

$ mpiexec -n 2 ./.venv/bin/python src/problems/topology/jax/solve_topopt_parallel.py --nx 64 --ny 32 --fixed_pad_cells 4 --load_pad_cells 4 --outer_maxit 12 --design_maxit 8 --quiet --json_out artifacts/raw_results/notebook_runs/topology_parallel_benchmark/parallel_reference.json


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


{'parallel_compliance': None,
 'parallel_outer_iterations': None,
 'parallel_result': 'max_outer_iterations',
 'serial_compliance': None,
 'serial_result': 'max_outer_iterations'}


This notebook keeps the cases intentionally small. For maintained fine-grid scaling and the published figures, use the canonical result pages instead of these lightweight notebook runs.
